# Trabalho Prático 1 - INF01017
## Pré-processamento de Dados
### Dataset: Ames Mutagenicity

**Objetivo:** Preparar os dados para modelagem através de técnicas de pré-processamento.

---

## 1. Imports e Configurações

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
import warnings

warnings.filterwarnings('ignore')

# Configurações
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## 2. Carregamento dos Dados

In [ ]:
# Carregar dataset
df = pd.read_csv('../data/raw/ames_mutagenicity_data.csv')

print(f"Shape original: {df.shape}")
df.head()

## 3. Remoção de Colunas de Metadados

In [ ]:
# Remover colunas de metadados
metadata_cols = ['Id', 'Name', 'CAS', 'SMILES RDKit']
cols_to_remove = [col for col in metadata_cols if col in df.columns]

print(f"Removendo colunas: {cols_to_remove}")
df_clean = df.drop(columns=cols_to_remove)

print(f"Shape após remoção: {df_clean.shape}")

## 4. Identificação de Colunas

In [ ]:
# Identificar tipos de colunas
target_col = 'Overall'
strain_cols = ['TA98', 'TA100', 'TA102', 'TA1535', 'TA1537']
partition_col = 'Partition'

# Opcionalmente, remover coluna de partição
if partition_col in df_clean.columns:
    df_clean = df_clean.drop(columns=[partition_col])
    print(f"Coluna '{partition_col}' removida.")

# Features
feature_cols = [col for col in df_clean.columns 
               if col not in [target_col] + strain_cols]

print(f"\nNúmero de features: {len(feature_cols)}")
print(f"Coluna alvo: {target_col}")

## 5. Tratamento de Valores Faltantes

In [ ]:
# Verificar valores faltantes
n_missing = df_clean.isnull().sum().sum()
print(f"Total de valores faltantes: {n_missing}")

if n_missing > 0:
    # Estratégia: remover colunas com >50% missing e linhas restantes
    threshold = 0.5
    missing_ratio = df_clean.isnull().sum() / len(df_clean)
    cols_to_drop = missing_ratio[missing_ratio > threshold].index.tolist()
    
    if cols_to_drop:
        df_clean = df_clean.drop(columns=cols_to_drop)
        print(f"Colunas removidas (>{threshold*100}% missing): {len(cols_to_drop)}")
    
    df_clean = df_clean.dropna()
    print(f"Shape após tratamento: {df_clean.shape}")
else:
    print("✓ Nenhum valor faltante encontrado!")

## 6. Remoção de Features com Baixa Variância

In [ ]:
# Atualizar lista de features
feature_cols = [col for col in df_clean.columns 
               if col not in [target_col] + strain_cols]

# Calcular variância
threshold_variance = 0.01
variances = df_clean[feature_cols].var()
low_variance_features = variances[variances < threshold_variance].index.tolist()

print(f"Features com variância < {threshold_variance}: {len(low_variance_features)}")

if low_variance_features:
    df_clean = df_clean.drop(columns=low_variance_features)
    print(f"Shape após remoção: {df_clean.shape}")

## 7. Normalização das Features

In [ ]:
# Atualizar lista de features novamente
feature_cols = [col for col in df_clean.columns 
               if col not in [target_col] + strain_cols]

# Normalização usando StandardScaler
scaler = StandardScaler()
df_normalized = df_clean.copy()

numeric_features = [col for col in feature_cols if col in df_clean.columns 
                   and df_clean[col].dtype in [np.float64, np.int64]]

df_normalized[numeric_features] = scaler.fit_transform(df_clean[numeric_features])

print(f"Features normalizadas: {len(numeric_features)}")
print("\nEstatísticas após normalização (primeiras 5 features):")
df_normalized[numeric_features[:5]].describe()

## 8. Divisão dos Dados

In [ ]:
# Separar features e target
X = df_normalized.drop(columns=[target_col])
y = df_normalized[target_col]

# Split: 70% treino, 10% validação, 20% teste
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.125, random_state=RANDOM_SEED, stratify=y_temp
)

print(f"Tamanho do conjunto de treino: {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Tamanho do conjunto de validação: {len(X_val)} ({len(X_val)/len(X)*100:.1f}%)")
print(f"Tamanho do conjunto de teste: {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)")

print("\nDistribuição das classes:")
print(f"Treino: {y_train.value_counts().to_dict()}")
print(f"Validação: {y_val.value_counts().to_dict()}")
print(f"Teste: {y_test.value_counts().to_dict()}")

## 9. Salvamento dos Dados Processados

In [ ]:
import os

# Criar DataFrames completos
train_df = X_train.copy()
train_df[target_col] = y_train

val_df = X_val.copy()
val_df[target_col] = y_val

test_df = X_test.copy()
test_df[target_col] = y_test

# Salvar
output_path = '../data/processed/'
os.makedirs(output_path, exist_ok=True)

train_df.to_csv(os.path.join(output_path, 'train.csv'), index=False)
val_df.to_csv(os.path.join(output_path, 'validation.csv'), index=False)
test_df.to_csv(os.path.join(output_path, 'test.csv'), index=False)

print(f"Dados processados salvos em: {output_path}")
print("✓ Arquivos criados: train.csv, validation.csv, test.csv")

## 10. Resumo do Pré-processamento

**Etapas realizadas:**

1. Remoção de colunas de metadados
2. Tratamento de valores faltantes
3. Remoção de features com baixa variância
4. Normalização das features (StandardScaler)
5. Divisão dos dados (treino/validação/teste)
6. Salvamento dos dados processados

**Próximo passo:** Spot-checking de algoritmos

---